# 05 — Executive Dashboard
**Financial Operations Analytics**

A single interactive Plotly view that consolidates the project's headline outputs:

- **Revenue trend + 12-month forecast** (with confidence band) — from NB 02
- **Churn-rate gauge** — from NB 03
- **Profit by segment** bar chart, **bottom 3 segments highlighted red** — from NB 04
- **At-risk customers table** — from NB 03
- **Region + Segment dropdown filters** that recompute the gauge, bar and table

The view is exported to a **standalone HTML file** (`dashboard/financial_ops_dashboard.html`)
that stays fully interactive offline — the two dropdowns are wired with a small embedded
script (Plotly served from CDN), so no Python server is required.


In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd().parent if Path.cwd().name in {"notebooks", "dashboard"} else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.viz import ACCENT, DANGER, PLOTLY_TEMPLATE

PROC = ROOT / "data" / "processed"
HTML_OUT = ROOT / "dashboard" / "financial_ops_dashboard.html"

mrr = pd.read_csv(PROC / "monthly_revenue.csv", index_col=0, parse_dates=True).iloc[:, 0]
mrr.name = "Revenue"
forecast = pd.read_csv(PROC / "revenue_forecast.csv", parse_dates=["Month"])
seg_df = pd.read_csv(PROC / "customer_segments.csv")
print("Loaded:", mrr.shape, forecast.shape, seg_df.shape)

## 1. Pre-compute filtered aggregates
For every **Region × Segment** combination we pre-compute the churn rate and at-risk
table; the profit-by-segment bar is computed per region (it *is* the segment
breakdown). These are embedded in the HTML so the dropdowns filter instantly client-
side.

In [ ]:
seg_order = ["Platinum", "Gold", "Silver", "Bronze", "Standard", "Basic", "Entry"]
seg_names = [s for s in seg_order if s in set(seg_df["Segment"])]
regions = ["All"] + sorted(seg_df["Region"].unique())
segments = ["All"] + seg_names

def bar_for_region(r):
    sub = seg_df if r == "All" else seg_df[seg_df["Region"] == r]
    # Average profit per customer keeps tier names and the red highlight consistent.
    prof = (sub.groupby("Segment")["MonthlyProfit"].mean()
            .reindex(seg_names).fillna(0))
    bottom3 = prof.sort_values(ascending=False).tail(3).index
    colors = [DANGER if s in bottom3 else ACCENT for s in seg_names]
    return [round(v, 2) for v in prof.tolist()], colors

def table_for(sub):
    a = (sub[sub["Churn Value"] == 0]
         .sort_values("ChurnProbability", ascending=False).head(8))
    return [a["CustomerID"].tolist(),
            a["Contract"].tolist(),
            a["Tenure Months"].tolist(),
            a["Monthly Charges"].round(2).tolist(),
            a["ChurnProbability"].round(3).tolist()]

DASH = {}
for r in regions:
    bar_vals, bar_cols = bar_for_region(r)
    for s in segments:
        sub = seg_df.copy()
        if r != "All":
            sub = sub[sub["Region"] == r]
        if s != "All":
            sub = sub[sub["Segment"] == s]
        churn = round(sub["Churn Value"].mean() * 100, 1) if len(sub) else 0.0
        DASH[f"{r}|{s}"] = {
            "churn": churn,
            "bar_vals": bar_vals,
            "bar_cols": bar_cols,
            "table": table_for(sub),
        }
print(f"Pre-computed {len(DASH)} region x segment states")

## 2. Build the figure

In [ ]:
base = DASH["All|All"]
fig = make_subplots(
    rows=3, cols=2,
    row_heights=[0.34, 0.30, 0.36],
    specs=[[{"type": "xy", "colspan": 2}, None],
           [{"type": "indicator"}, {"type": "xy"}],
           [{"type": "table", "colspan": 2}, None]],
    subplot_titles=("Monthly Revenue — history & 12-month forecast",
                    "Churn rate", "Avg profit / customer by segment (bottom 3 red)",
                    "Top at-risk customers"),
    vertical_spacing=0.10, horizontal_spacing=0.12)

# --- Row 1: revenue history + forecast + CI band ---
fig.add_trace(go.Scatter(x=mrr.index, y=mrr.values, name="History",
                         line=dict(color=ACCENT, width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=forecast["Month"], y=forecast["Forecast"], name="Forecast",
                         line=dict(color=DANGER, width=2.5, dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=forecast["Month"], y=forecast["Upper"], name="Upper",
                         line=dict(width=0), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=forecast["Month"], y=forecast["Lower"], name="90% CI",
                         line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(192,80,77,0.18)"), row=1, col=1)

# --- Row 2 col 1: churn gauge ---
overall_churn = round(seg_df["Churn Value"].mean() * 100, 1)
fig.add_trace(go.Indicator(
    mode="gauge+number", value=base["churn"],
    number={"suffix": "%"},
    gauge={"axis": {"range": [0, 60]},
           "bar": {"color": ACCENT},
           "steps": [{"range": [0, 25], "color": "#E8F0E5"},
                     {"range": [25, 40], "color": "#FBEED7"},
                     {"range": [40, 60], "color": "#F4D6D4"}],
           "threshold": {"line": {"color": DANGER, "width": 3}, "value": overall_churn}},
    title={"text": "Churn rate"}), row=2, col=1)

# --- Row 2 col 2: profit by segment ---
fig.add_trace(go.Bar(x=seg_names, y=base["bar_vals"],
                     marker_color=base["bar_cols"], showlegend=False,
                     name="Profit"), row=2, col=2)

# --- Row 3: at-risk table ---
fig.add_trace(go.Table(
    header=dict(values=["Customer", "Contract", "Tenure", "Monthly $", "Churn Prob"],
                fill_color=ACCENT, font=dict(color="white", size=12), align="left"),
    cells=dict(values=base["table"], align="left",
               fill_color="#F5F8FB", height=24)), row=3, col=1)

fig.update_layout(
    template=PLOTLY_TEMPLATE, height=980, width=1150,
    title=dict(text="<b>Financial Operations Analytics — Executive Dashboard</b>",
               x=0.5, font=dict(size=22)),
    margin=dict(t=120, b=40, l=40, r=40),
    legend=dict(orientation="h", y=1.02, x=0.0))
fig.update_yaxes(title_text="Revenue ($)", row=1, col=1)
fig.update_yaxes(title_text="Avg profit / cust ($)", row=2, col=2)
fig.show()

## 3. Export to standalone interactive HTML
The two dropdown filters are added with an embedded post-script that calls
`Plotly.restyle` on change — the pre-computed `DASH` table is serialized into the page,
so filtering works entirely client-side with no server.

In [ ]:
POST_JS = """
var gd = document.getElementsByClassName('plotly-graph-div')[0];
var DASH = __DASH__;
var REGIONS = __REGIONS__;
var SEGMENTS = __SEGMENTS__;
var GAUGE = 4, BAR = 5, TABLE = 6;   // trace indices

function mkSelect(id, label, opts) {
  var wrap = document.createElement('div');
  wrap.style.cssText = 'display:inline-block;margin:0 18px 0 0;font-family:DejaVu Sans,Arial,sans-serif';
  var lab = document.createElement('label');
  lab.textContent = label + ': ';
  lab.style.cssText = 'font-weight:bold;color:#2E5A88;margin-right:6px';
  var sel = document.createElement('select');
  sel.id = id;
  sel.style.cssText = 'padding:5px 8px;border:1px solid #2E5A88;border-radius:5px;font-size:14px';
  opts.forEach(function(o){ var e=document.createElement('option'); e.value=o; e.text=o; sel.appendChild(e); });
  sel.addEventListener('change', applyFilter);
  wrap.appendChild(lab); wrap.appendChild(sel);
  return wrap;
}

function applyFilter() {
  var r = document.getElementById('region-sel').value;
  var s = document.getElementById('segment-sel').value;
  var d = DASH[r + '|' + s] || DASH['All|All'];
  Plotly.restyle(gd, {'value': [d.churn]}, [GAUGE]);
  Plotly.restyle(gd, {'y': [d.bar_vals], 'marker.color': [d.bar_cols]}, [BAR]);
  Plotly.restyle(gd, {'cells.values': [d.table]}, [TABLE]);
}

var bar = document.createElement('div');
bar.style.cssText = 'text-align:center;margin:14px 0 4px 0';
bar.appendChild(mkSelect('region-sel', 'Region', REGIONS));
bar.appendChild(mkSelect('segment-sel', 'Segment', SEGMENTS));
gd.parentNode.insertBefore(bar, gd);
"""

post = (POST_JS
        .replace("__DASH__", json.dumps(DASH))
        .replace("__REGIONS__", json.dumps(regions))
        .replace("__SEGMENTS__", json.dumps(segments)))

HTML_OUT.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(HTML_OUT, include_plotlyjs="cdn", full_html=True,
               post_script=post, config={"displayModeBar": True})
print(f"Exported standalone dashboard -> {HTML_OUT.relative_to(ROOT)}")
print(f"Size: {HTML_OUT.stat().st_size/1024:.0f} KB")

## Done
Open `dashboard/financial_ops_dashboard.html` in any browser. Use the **Region** and
**Segment** dropdowns to filter the churn gauge, profit-by-segment bar (bottom 3 always
flagged red) and the at-risk customer table. The revenue trend and 12-month forecast
provide the company-wide top line.
